# YOLO11 Training on Oxford-IIIT Pet Dataset (Cats)

This notebook trains a YOLO11n model on the cat detection dataset from the Oxford-IIIT Pet Dataset.

## Configuration

In [1]:
import os
import shutil
import time
from pathlib import Path
from typing import Any, Self

import albumentations as A
import kagglehub
import polars as pl
import torch
from pydantic import BaseModel, Field
from ultralytics.models import YOLO

# Training configuration
DEVICE = [0, 1]  # If on Kaggle with T4x2, use multiple GPUs
MODEL = "yolo11n.pt"  # YOLO11 nano model
EPOCHS = 100
IMAGE_SIZE = 640

# Which presets to train (set to None to train all)
PRESETS_TO_TRAIN: list[str] | None = ["baseline", "all", "geometric", "jitter", "mosaic", "no_mosaic"] # e.g., ["baseline", "all", "geometric"]

# reference: https://docs.ultralytics.com/zh/guides/yolo-data-augmentation/
class Augmentation(BaseModel):
    # Geometric augmentations
    degrees: float = 0.0  # Rotation range (±degrees)
    translate: float = 0.0  # Translation (±fraction)
    scale: float = 0.0  # Scale variation (±fraction)
    shear: float = 0.0  # Shear angle (±degrees)
    perspective: float = 0.0  # Perspective distortion
    fliplr: float = 0.0  # Horizontal flip probability
    flipud: float = 0.0  # Vertical flip probability
    # Color/HSV augmentations
    hsv_h: float = 0.0  # Hue variation
    hsv_s: float = 0.0  # Saturation variation
    hsv_v: float = 0.0  # Brightness variation
    # Advanced augmentations
    mosaic: float = 0.0  # Mosaic augmentation (4-image combine)
    mixup: float = 0.0  # MixUp regularization
    close_mosaic: int = 0  # Disable mosaic in last N epochs
    augmentations: list[Any] = Field(default_factory=list)

    def combine(self, other: Self) -> "Augmentation":
        """Combine two Augmentation configs, taking max/non-zero values."""
        return Augmentation(
            # Geometric: take max
            degrees=max(self.degrees, other.degrees),
            translate=max(self.translate, other.translate),
            scale=max(self.scale, other.scale),
            shear=max(self.shear, other.shear),
            perspective=max(self.perspective, other.perspective),
            fliplr=max(self.fliplr, other.fliplr),
            flipud=max(self.flipud, other.flipud),
            # HSV: take max
            hsv_h=max(self.hsv_h, other.hsv_h),
            hsv_s=max(self.hsv_s, other.hsv_s),
            hsv_v=max(self.hsv_v, other.hsv_v),
            # Mosaic: take max
            mosaic=max(self.mosaic, other.mosaic),
            mixup=max(self.mixup, other.mixup),
            close_mosaic=max(self.close_mosaic, other.close_mosaic),
            # Augmentations: concatenate lists
            augmentations=[*self.augmentations, *other.augmentations],
        )

    def __or__(self, other: Self) -> "Augmentation":
        """Combine using | operator: aug1 | aug2"""
        return self.combine(other)


# Shared values (used across all presets for consistency)
_GEOMETRIC = Augmentation(
    degrees=15.0, translate=0.1, scale=0.5, shear=2.0, perspective=0.0005, fliplr=0.5
)
_HSV = Augmentation(hsv_h=0.02, hsv_s=0.7, hsv_v=0.4)
_MOSAIC = Augmentation(mosaic=1.0, mixup=0.15, close_mosaic=10)
_BLUR = Augmentation(
    augmentations=[
        A.OneOf([A.MotionBlur(blur_limit=7), A.GaussianBlur(blur_limit=7)], p=0.3)
    ]
)
_NOISE = Augmentation(augmentations=[A.GaussNoise(std_range=(0.1, 0.3), p=0.2)])

AUGMENT_PRESETS: dict[str, Augmentation] = {
    "baseline": Augmentation(),           # Lower bound
    "all": _GEOMETRIC | _HSV | _MOSAIC,   # Upper bound (no alb for DDP)
    "geometric": _GEOMETRIC,              # Test geometric alone
    "jitter": _HSV,                       # Test color alone
    "mosaic": _MOSAIC,                    # Test mosaic alone
    "no_mosaic": _GEOMETRIC | _HSV,       # Most interesting: mosaic is often the biggest contributor
}

# Determine which presets to train
presets_to_run = PRESETS_TO_TRAIN or list(AUGMENT_PRESETS.keys())
print(f"Will train {len(presets_to_run)} presets: {presets_to_run}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Will train 6 presets: ['baseline', 'all', 'geometric', 'jitter', 'mosaic', 'no_mosaic']


## Dataset Setup

Download and prepare the YOLO-formatted dataset.

In [2]:
slug = "inogai/of3t-cats-yolo"

# Download and copy dataset to working directory (Kaggle input is read-only)
src_path = Path(kagglehub.dataset_download("inogai/of3t-cats-yolo"))
work_path = Path("/kaggle/working/datasets") / slug.split("/")[-1]
_ = shutil.copytree(src_path, work_path, dirs_exist_ok=True)

dataset_path = work_path / "data.yaml"
print(f"Dataset ready at: {dataset_path}")

Dataset ready at: /kaggle/working/datasets/of3t-cats-yolo/data.yaml


## Model Training

Initialize and train the YOLO11 model.

In [3]:
results_summary: dict[str, dict[str, float]] = {}

for preset_name in presets_to_run:
    print(f"\n{'='*60}")
    print(f"Training preset: {preset_name} ({presets_to_run.index(preset_name)+1}/{len(presets_to_run)})")
    print(f"{'='*60}\n")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    
    # Fresh model for each run (fair comparison)
    model = YOLO(MODEL, task="detect")

    aug = AUGMENT_PRESETS[preset_name]
    has_alb = len(aug.augmentations) > 0
    device = 0 if has_alb else DEVICE
    
    start_time = time.time()
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        device=device,
        project="runs/ablation",
        name=preset_name,
        exist_ok=True,
        **aug.model_dump(),
    )
    train_time = time.time() - start_time
    
    # Validate and collect metrics
    metrics = model.val()
    
    results_summary[preset_name] = {
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "train_time_min": round(train_time / 60, 2),
    }
    
    print(f"\n✓ {preset_name}: mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

print("\n" + "="*60)
print("All training complete!")
print("="*60)


Training preset: baseline (1/6)

Ultralytics 8.3.235 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, augmentations=[], auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/datasets/of3t-cats-yolo/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scal

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        474        474       0.81      0.833      0.894      0.743
            Abyssinian         40         40      0.871        0.9      0.942      0.791
                Bengal         40         40      0.698      0.636      0.759      0.649
                Birman         40         40      0.833      0.747      0.876      0.756
                Bombay         37         37      0.942      0.883      0.963      0.859
     British_Shorthair         40         40      0.768      0.745      0.834      0.722
          Egyptian_Mau         37         37      0.806      0.919       0.94      0.764
            Maine_Coon         40         40       0.75       0.75      0.838      0.646
               Persian         40         40      0.837        0.8      0.897      0.748
               Ragdoll         40         40      0.748      0.825      0.874      0.676
          Russian_Blue         40         40      0.662       0.85      0.845      0.687
               Siames

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        474        474      0.811      0.833      0.894      0.743
            Abyssinian         40         40      0.871        0.9      0.942      0.791
                Bengal         40         40        0.7      0.642      0.758      0.648
                Birman         40         40      0.832      0.745      0.876      0.756
                Bombay         37         37      0.942      0.882      0.963      0.859
     British_Shorthair         40         40      0.768      0.745      0.833      0.718
          Egyptian_Mau         37         37      0.807      0.919       0.94      0.765
            Maine_Coon         40         40      0.754       0.75      0.836      0.651
               Persian         40         40      0.837        0.8      0.896      0.747
               Ragdoll         40         40      0.749      0.825      0.874      0.671
          Russian_Blue         40         40      0.662       0.85      0.847      0.693
               Siames

...

                   all        474        474      0.914      0.919      0.956      0.803
            Abyssinian         40         40       0.95        0.9       0.99      0.799
                Bengal         40         40      0.824      0.875      0.943      0.794
                Birman         40         40      0.831      0.925      0.907       0.77
                Bombay         37         37      0.972      0.938      0.966      0.845
     British_Shorthair         40         40      0.941      0.875      0.949      0.848
          Egyptian_Mau         37         37      0.915      0.869      0.976      0.764
            Maine_Coon         40         40      0.922       0.95       0.94      0.771
               Persian         40         40      0.944      0.975      0.986      0.839
               Ragdoll         40         40      0.942      0.825      0.921      0.788
          Russian_Blue         40         40      0.817      0.894      0.909      0.795
               Siames

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        474        474      0.909      0.911      0.952      0.783
            Abyssinian         40         40       0.88      0.925       0.96      0.834
                Bengal         40         40      0.964        0.8       0.93      0.736
                Birman         40         40      0.812      0.865       0.89      0.696
                Bombay         37         37      0.943      0.895      0.969       0.82
     British_Shorthair         40         40      0.952      0.875      0.952       0.82
          Egyptian_Mau         37         37      0.876      0.973      0.979       0.81
            Maine_Coon         40         40      0.905      0.955      0.949      0.751
               Persian         40         40      0.962        0.9      0.984      0.856
               Ragdoll         40         40      0.845        0.8      0.899      0.733
          Russian_Blue         40         40      0.861      0.975      0.923       0.74
               Siames

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        474        474      0.909      0.912      0.952      0.783
            Abyssinian         40         40       0.88      0.925       0.96      0.833
                Bengal         40         40      0.963        0.8       0.93      0.737
                Birman         40         40      0.814      0.873       0.89      0.695
                Bombay         37         37      0.943      0.896      0.969      0.818
     British_Shorthair         40         40      0.952      0.875      0.952       0.82
          Egyptian_Mau         37         37      0.876      0.973      0.979      0.807
            Maine_Coon         40         40      0.905      0.956      0.949      0.748
               Persian         40         40      0.961        0.9      0.984      0.856
               Ragdoll         40         40      0.847        0.8      0.899      0.733
          Russian_Blue         40         40      0.861      0.975      0.923      0.742
               Siames

## Model Validation

Evaluate the trained model on the validation set.

In [4]:
# Create results DataFrame
df = pl.DataFrame([
    {"preset": k, **v} for k, v in results_summary.items()
]).sort("mAP50-95", descending=True)

print("\n=== Ablation Study Results ===\n")
print(df)

# Save to CSV
df.write_csv("ablation_results.csv")
print("\nResults saved to ablation_results.csv")


=== Ablation Study Results ===

shape: (6, 6)
┌───────────┬──────────┬──────────┬───────────┬──────────┬────────────────┐
│ preset    ┆ mAP50    ┆ mAP50-95 ┆ precision ┆ recall   ┆ train_time_min │
│ ---       ┆ ---      ┆ ---      ┆ ---       ┆ ---      ┆ ---            │
│ str       ┆ f64      ┆ f64      ┆ f64       ┆ f64      ┆ f64            │
╞═══════════╪══════════╪══════════╪═══════════╪══════════╪════════════════╡
│ mosaic    ┆ 0.955709 ┆ 0.80322  ┆ 0.913834  ┆ 0.918847 ┆ 38.94          │
│ all       ┆ 0.966757 ┆ 0.799509 ┆ 0.942774  ┆ 0.922883 ┆ 47.33          │
│ geometric ┆ 0.952746 ┆ 0.787451 ┆ 0.910668  ┆ 0.924705 ┆ 38.83          │
│ no_mosaic ┆ 0.951647 ┆ 0.782604 ┆ 0.909376  ┆ 0.911661 ┆ 43.46          │
│ baseline  ┆ 0.894256 ┆ 0.743468 ┆ 0.810812  ┆ 0.833205 ┆ 33.96          │
│ jitter    ┆ 0.877436 ┆ 0.720805 ┆ 0.854839  ┆ 0.794319 ┆ 36.29          │
└───────────┴──────────┴──────────┴───────────┴──────────┴────────────────┘

Results saved to ablation_results.csv


## Clean Up

Remove the dataset from the working directory (output) to free up space.

In [5]:
_ = shutil.rmtree("/kaggle/working/datasets")
_ = os.remove("/kaggle/working/yolo11n.pt")